In [ ]:
# Notebook: Frequency Analysis & On-Device Simulation (Cortex-M4) — Improved
# Python >= 3.9
# Requires: numpy, pandas, pywt, matplotlib, seaborn, scikit-learn, scipy, (optional) torch

# =========================
# Cell 0: Environment & Config
# =========================
import os
from pathlib import Path
from typing import Tuple, Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pywt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from scipy import signal

sns.set_context("talk")
sns.set_style("whitegrid")

SEED = 42
rng = np.random.default_rng(SEED)

CONFIG = {
    # Data & processing
    "data_dir": "data",
    "proc_dir": "data/processed",
    "results_dir": "results",
    "fig_dir": "results/figures",
    "out_len": 256,              # resampled spectrum length
    "downsample_to": None,       # e.g., 128 or None
    "standardize": True,         # z-score with train stats
    "pca_components": None,      # e.g., 128 or None (applied to raw only)
    "val_ratio": 0.2,

    # DWT
    "dwt_wavelet": "db4",
    "dwt_level": 2,
    "cache_dwt": True,

    # Visualization
    "scalogram_num": 6,          # how many validation samples to visualize
    "scalogram_scales": 64,
    "scalogram_wavelet": "morl",
    "topk_importance": 20,

    # Baseline models
    "run_rf": True,
    "rf_n_estimators": 500,
    "rf_n_jobs": -1,

    "run_mlp": False,            # optional quick MLP on combined features
    "mlp_hidden": 256,
    "mlp_epochs": 300,
    "mlp_lr": 5e-4,
    "mlp_wd": 1e-3,

    # MCU simulation (Cortex-M4 defaults; tune per board)
    "mcu_clock_hz": 96e6,
    "mcu_voltage_v": 3.3,
    "mcu_current_a": 0.035,

    # Reproducibility
    "seed": SEED,
}

DATA_DIR = Path(CONFIG["data_dir"])
PROC_DIR = Path(CONFIG["proc_dir"])
RES_DIR = Path(CONFIG["results_dir"])
FIG_DIR = Path(CONFIG["fig_dir"])
for p in [PROC_DIR, RES_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# =========================
# Cell 1: Data Loading with Fallback
# =========================
def parse_gain_columns(df: pd.DataFrame, family_prefix: str) -> Tuple[np.ndarray, List[str]]:
    """Parse rx_gain columns and return (freqs, column_names) sorted by frequency."""
    cols, freqs = [], []
    for c in df.columns:
        if c.startswith(family_prefix + "_f_"):
            f = c.split("_f_")[-1]
            if f.isdigit():
                freqs.append(float(f)); cols.append(c)
    order = np.argsort(freqs)
    return np.asarray(freqs, dtype=float)[order], [cols[i] for i in order]


def select_gain_family(df: pd.DataFrame) -> Tuple[str, np.ndarray, List[str]]:
    """Choose gain family with lower zero/NaN ratio across rows."""
    f1, c1 = parse_gain_columns(df, "rx_gain_50")
    f2, c2 = parse_gain_columns(df, "rx_gain_1M")

    def score(cols):
        if len(cols) == 0: return np.inf
        vals = df[cols].replace([np.inf, -np.inf], np.nan)
        return float(((vals == 0) | vals.isna()).sum(axis=1).mean())

    s1, s2 = score(c1), score(c2)
    if s1 < s2 and len(c1) > 0:
        return "rx_gain_50", f1, c1
    return "rx_gain_1M", f2, c2


def resample_spectrum(row_vals: np.ndarray, in_freqs: np.ndarray, out_len: int = 256) -> np.ndarray:
    """Linear interpolation to fixed length; expects dB-domain magnitudes."""
    fmin, fmax = float(in_freqs.min()), float(in_freqs.max())
    f_out = np.linspace(fmin, fmax, out_len, dtype=float)
    return np.interp(f_out, in_freqs, row_vals.astype(float))


def load_or_build_processed(out_len: int = 256) -> Tuple[np.ndarray, np.ndarray]:
    """Load processed X(out_len) and y, or build from all_measurements.csv."""
    x_csv = PROC_DIR / "ibc_processed.csv"
    y_csv = DATA_DIR / "labels_filtered.csv"
    if x_csv.exists() and y_csv.exists():
        X = pd.read_csv(x_csv).values.astype(np.float32)
        y = pd.read_csv(y_csv)["subject_id"].astype(int).values
        return X, y

    raw_csvs = list(DATA_DIR.glob("all_measurements.csv"))
    if not raw_csvs:
        raise FileNotFoundError("Provide data/processed/ibc_processed.csv or data/all_measurements.csv")

    df = pd.read_csv(raw_csvs[0])
    assert "subject_id" in df.columns, "subject_id missing in all_measurements.csv"
    family, freqs, cols = select_gain_family(df)
    vals = df[cols].replace([np.inf, -np.inf], np.nan)
    mask = ~vals.isna().any(axis=1)
    df = df.loc[mask].reset_index(drop=True)
    vals = vals.loc[mask].reset_index(drop=True)
    y = df["subject_id"].astype(int).values

    X = np.vstack([resample_spectrum(vals.iloc[i].values, freqs, out_len=out_len)
                   for i in range(len(vals))]).astype(np.float32)

    pd.DataFrame(X, columns=[f"f{i}" for i in range(X.shape[1])]).to_csv(x_csv, index=False)
    pd.DataFrame({"subject_id": y}).to_csv(y_csv, index=False)
    return X, y


X_raw, y = load_or_build_processed(out_len=CONFIG["out_len"])
print("Processed:", X_raw.shape, y.shape)

# =========================
# Cell 2: Split & Preprocess (leakage-free)
# =========================
sss = StratifiedShuffleSplit(n_splits=1, test_size=CONFIG["val_ratio"], random_state=CONFIG["seed"])
tr_idx, va_idx = next(sss.split(X_raw, y))
X_tr_raw, X_va_raw = X_raw[tr_idx], X_raw[va_idx]
y_tr, y_va = y[tr_idx], y[va_idx]
print("Split:", X_tr_raw.shape, X_va_raw.shape, len(np.unique(y_tr)), len(np.unique(y_va)))

# Optional downsample (e.g., 256 -> 128) BEFORE standardization
def maybe_downsample(X: np.ndarray, to_len: Optional[int]) -> np.ndarray:
    if to_len is None or to_len == X.shape[1]:
        return X
    # Use Fourier resampling for smooth decimation
    return signal.resample(X, num=to_len, axis=1)

X_tr_ds = maybe_downsample(X_tr_raw, CONFIG["downsample_to"])
X_va_ds = maybe_downsample(X_va_raw, CONFIG["downsample_to"])
print("After downsample:", X_tr_ds.shape, X_va_ds.shape)

# Standardization with train stats
if CONFIG["standardize"]:
    scaler = StandardScaler(with_mean=True, with_std=True)
    X_tr = scaler.fit_transform(X_tr_ds)
    X_va = scaler.transform(X_va_ds)
else:
    X_tr, X_va = X_tr_ds, X_va_ds

# Optional PCA on RAW (retain DWT over standardized raw)
def maybe_pca_fit_transform(Xtr: np.ndarray, Xva: np.ndarray, n_comp: Optional[int]):
    if n_comp is None:
        return Xtr, Xva, None
    pca = PCA(n_components=n_comp, random_state=CONFIG["seed"])
    return pca.fit_transform(Xtr), pca.transform(Xva), pca

X_tr_raw_pca, X_va_raw_pca, pca_model = maybe_pca_fit_transform(X_tr, X_va, CONFIG["pca_components"])
print("Raw/PCA shapes:", X_tr.shape, X_va.shape, X_tr_raw_pca.shape, X_va_raw_pca.shape)

# =========================
# Cell 3: DWT(db4, L2) feature extraction (with caching)
# =========================
def dwt_stats(X: np.ndarray, wavelet: str = "db4", level: int = 2) -> np.ndarray:
    """Compute DWT stats per sample: energy, entropy, mean, std for each coeff array."""
    feats = []
    for row in X:
        coeffs = pywt.wavedec(row, wavelet, level=level, mode="periodization")
        fvec = []
        for c in coeffs:
            e = float(np.sum(c**2))
            p = (c**2) / (e + 1e-12)
            ent = float(-np.sum(p * np.log2(np.clip(p, 1e-12, 1.0))))
            fvec.extend([e, ent, float(np.mean(c)), float(np.std(c))])
        feats.append(fvec)
    return np.asarray(feats, dtype=np.float32)

def dwt_cached(Xtr: np.ndarray, Xva: np.ndarray, wavelet: str, level: int) -> Tuple[np.ndarray, np.ndarray]:
    key = f"dwt_{wavelet}_L{level}_std{int(CONFIG['standardize'])}_len{Xtr.shape[1]}"
    ftr = PROC_DIR / f"{key}_train.npy"
    fva = PROC_DIR / f"{key}_val.npy"
    if CONFIG["cache_dwt"] and ftr.exists() and fva.exists():
        return np.load(ftr), np.load(fva)
    X_tr_dwt = dwt_stats(Xtr, wavelet=wavelet, level=level)
    X_va_dwt = dwt_stats(Xva, wavelet=wavelet, level=level)
    if CONFIG["cache_dwt"]:
        np.save(ftr, X_tr_dwt); np.save(fva, X_va_dwt)
    return X_tr_dwt, X_va_dwt

X_tr_dwt, X_va_dwt = dwt_cached(X_tr, X_va, CONFIG["dwt_wavelet"], CONFIG["dwt_level"])
print("DWT shapes:", X_tr_dwt.shape, X_va_dwt.shape)

# Combined features (raw/PCA + DWT)
X_tr_comb = np.hstack([X_tr_raw_pca, X_tr_dwt])
X_va_comb = np.hstack([X_va_raw_pca, X_va_dwt])
print("Combined shapes:", X_tr_comb.shape, X_va_comb.shape)

# =========================
# Cell 4: CWT Scalogram Grid
# =========================
def cwt_scalogram(x: np.ndarray, wavelet: str = "morl", num_scales: int = 64) -> np.ndarray:
    scales = np.linspace(1, num_scales, num_scales)
    coef, freqs = pywt.cwt(x, scales, wavelet)
    power = np.abs(coef)**2
    return power.astype(np.float32)

def save_scalogram_grid(Xv: np.ndarray, k: int, wavelet: str, num_scales: int, out_path: Path):
    idx = rng.choice(np.arange(Xv.shape[0]), size=min(k, Xv.shape[0]), replace=False)
    n = len(idx); cols = min(3, n); rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3.2*rows), squeeze=False)
    for ax, i in zip(axes.flat, idx):
        S = cwt_scalogram(Xv[i], wavelet=wavelet, num_scales=num_scales)
        sns.heatmap(S, cmap="magma", cbar=False, ax=ax)
        ax.set_title(f"val idx {i}")
        ax.set_xlabel("Freq idx"); ax.set_ylabel("Scales")
    for ax in axes.flat[n:]:
        ax.axis("off")
    fig.suptitle("CWT Scalogram Grid")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

save_scalogram_grid(X_va, CONFIG["scalogram_num"], CONFIG["scalogram_wavelet"],
                    CONFIG["scalogram_scales"], FIG_DIR / "scalogram_grid.png")

# =========================
# Cell 5: RF Feature Importance on DWT stats
# =========================
if CONFIG["run_rf"]:
    rf = RandomForestClassifier(
        n_estimators=CONFIG["rf_n_estimators"], max_depth=None,
        random_state=SEED, n_jobs=CONFIG["rf_n_jobs"]
    )
    rf.fit(X_tr_dwt, y_tr)
    imp = rf.feature_importances_
    order = np.argsort(imp)[::-1]
    topk = CONFIG["topk_importance"]

    # Bar plot
    plt.figure(figsize=(10, 4))
    plt.bar(np.arange(min(topk, len(imp))), imp[order][:topk])
    plt.title(f"Top-{topk} DWT Feature Importances (RF Gini)")
    plt.xlabel("DWT feature index (energy/entropy/mean/std per coeff)")
    plt.ylabel("Gini importance")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "dwt_importances_topk.png", dpi=200)
    plt.close()

    # CSV of importances
    pd.DataFrame({
        "feat_idx": order,
        "importance": imp[order]
    }).to_csv(RES_DIR / "dwt_feature_importances.csv", index=False)

# =========================
# Cell 6: MCU On-Device Simulation (Cortex-M4) — Stage-wise
# =========================
def cycles_zscore(n: int, d: int) -> int:
    # simple model: subtract mean, divide std, load/store per element
    C_ADD, C_DIV, C_MEM = 1, 3, 2
    return int(n * d * (C_ADD + C_DIV + C_MEM))

def cycles_dwt_db4_l2(n: int, d: int) -> int:
    # first-order estimator: lifting-like passes across ~2d with aggregated constants
    C_MUL, C_ADD, C_MEM = 1, 1, 2
    a = 20  # aggregated factor (tunable)
    dwt = int(a * 2 * d * (C_MUL + C_ADD + C_MEM))
    stats = int(10 * d * (C_ADD + C_MUL + C_MEM))  # energy/entropy/mean/std
    return (dwt + stats) * n

def time_energy(cycles: int, f_hz: float, v: float, i: float) -> Tuple[float, float]:
    t = cycles / f_hz
    e = v * i * t
    return t, e

def simulate_pipeline(n_samples: int, d_spec: int, clk: float, v: float, i: float) -> Dict[str, float]:
    c_z = cycles_zscore(n_samples, d_spec) if CONFIG["standardize"] else 0
    c_dwt = cycles_dwt_db4_l2(n_samples, d_spec)
    c_tot = c_z + c_dwt
    t, e = time_energy(c_tot, clk, v, i)
    return {
        "cycles_total": float(c_tot),
        "time_s": float(t),
        "energy_J": float(e),
        "cycles_zscore": float(c_z),
        "cycles_dwt": float(c_dwt),
        "clock_hz": float(clk),
        "voltage_v": float(v),
        "current_a": float(i),
        "n_samples": int(n_samples),
        "d_spec": int(d_spec),
    }

mcu = simulate_pipeline(
    n_samples=1,
    d_spec=X_tr.shape[1],
    clk=CONFIG["mcu_clock_hz"],
    v=CONFIG["mcu_voltage_v"],
    i=CONFIG["mcu_current_a"],
)
print("MCU Simulation (1 sample):", mcu)

# Also simulate a small batch (e.g., 50 samples)
mcu_batch = simulate_pipeline(
    n_samples=50,
    d_spec=X_tr.shape[1],
    clk=CONFIG["mcu_clock_hz"],
    v=CONFIG["mcu_voltage_v"],
    i=CONFIG["mcu_current_a"],
)

# =========================
# Cell 7: Lightweight Baselines (RF combined, optional MLP)
# =========================
results = []

if CONFIG["run_rf"]:
    clf = RandomForestClassifier(n_estimators=CONFIG["rf_n_estimators"], random_state=SEED, n_jobs=CONFIG["rf_n_jobs"])
    clf.fit(X_tr_comb, y_tr)
    y_pred = clf.predict(X_va_comb)
    acc = accuracy_score(y_va, y_pred)
    print("RF(combined) val accuracy:", acc)
    results.append({"model": "rf_combined", "val_acc": float(acc)})

if CONFIG["run_mlp"]:
    try:
        import torch, torch.nn as nn, torch.nn.functional as F
        from torch.utils.data import TensorDataset, DataLoader
        DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

        class MLP(nn.Module):
            def __init__(self, in_dim, n_classes, hidden=256, p=0.3):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(in_dim, hidden),
                    nn.ReLU(inplace=True),
                    nn.Dropout(p),
                    nn.Linear(hidden, hidden),
                    nn.ReLU(inplace=True),
                    nn.Dropout(p),
                    nn.Linear(hidden, n_classes),
                )
            def forward(self, x): return self.net(x)

        C = len(np.unique(y_tr))
        D = X_tr_comb.shape[1]
        model = MLP(D, C, hidden=CONFIG["mlp_hidden"], p=0.3).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["mlp_lr"], weight_decay=CONFIG["mlp_wd"])
        crit = nn.CrossEntropyLoss()

        Xt = torch.from_numpy(X_tr_comb).float().to(DEVICE)
        yt = torch.from_numpy(pd.factorize(y_tr)[0]).long().to(DEVICE)
        Xv = torch.from_numpy(X_va_comb).float().to(DEVICE)
        yv = torch.from_numpy(pd.factorize(y_va, sort=True)[0]).long().to(DEVICE)
        # Align val encoding to train classes
        # Build mapping from original labels to 0..C-1 using train labels
        classes = np.unique(y_tr)
        to_index = {int(c): i for i, c in enumerate(classes)}
        y_tr_enc = np.vectorize(lambda t: to_index[int(t)])(y_tr)
        y_va_enc = np.vectorize(lambda t: to_index[int(t)])(y_va)
        yt = torch.from_numpy(y_tr_enc).long().to(DEVICE)
        yv = torch.from_numpy(y_va_enc).long().to(DEVICE)

        dl = DataLoader(TensorDataset(Xt, yt), batch_size=64, shuffle=True, drop_last=False)
        best_val = 1e9; best_state=None
        for epoch in range(1, CONFIG["mlp_epochs"]+1):
            model.train(); run=0.0
            for xb, yb in dl:
                opt.zero_grad(); logits = model(xb); loss = crit(logits, yb)
                loss.backward(); opt.step(); run += loss.item() * xb.size(0)
            tr_loss = run / len(dl.dataset)
            model.eval()
            with torch.no_grad():
                va_loss = crit(model(Xv), yv).item()
                preds = model(Xv).argmax(dim=1).cpu().numpy()
                va_acc = (preds == y_va_enc).mean()
            if va_loss < best_val:
                best_val = va_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if epoch % 50 == 0 or epoch == 1:
                print(f"[MLP {epoch:03d}] tr={tr_loss:.4f} val={va_loss:.4f} acc={va_acc:.4f}")
        if best_state is not None:
            model.load_state_dict(best_state)
        with torch.no_grad():
            preds = model(Xv).argmax(dim=1).cpu().numpy()
            acc = (preds == y_va_enc).mean()
        print("MLP(combined) val accuracy:", acc)
        results.append({"model": "mlp_combined", "val_acc": float(acc)})
    except Exception as e:
        print("MLP failed:", e)

# =========================
# Cell 8: Save Reports
# =========================
report = {
    "n_train": int(X_tr.shape[0]), "n_val": int(X_va.shape[0]),
    "d_raw": int(X_tr.shape[1]), "d_dwt": int(X_tr_dwt.shape[1]),
    "d_combined": int(X_tr_comb.shape[1]),
    "standardize": bool(CONFIG["standardize"]),
    "downsample_to": int(CONFIG["downsample_to"]) if CONFIG["downsample_to"] is not None else None,
    "pca_components": int(CONFIG["pca_components"]) if CONFIG["pca_components"] is not None else None,
    "dwt_wavelet": CONFIG["dwt_wavelet"], "dwt_level": int(CONFIG["dwt_level"]),
    "rf_ran": bool(CONFIG["run_rf"]), "mlp_ran": bool(CONFIG["run_mlp"]),
    "mcu_sim_1": mcu, "mcu_sim_50": mcu_batch,
}
pd.DataFrame(results).to_csv(RES_DIR / "baseline_results.csv", index=False)
pd.Series(report, dtype=object).to_json(RES_DIR / "ondevice_simulation_report.json", indent=2)

print("Saved:")
print(" - Figures: scalogram_grid.png, dwt_importances_topk.png")
print(" - Reports: dwt_feature_importances.csv, baseline_results.csv, ondevice_simulation_report.json")
